# 🌐 Tanglish DistilGPT2 — Fine-Tuning on Kaggle

**GPU runtime required.** Enable via *Settings → Accelerator → GPU T4 x2 or P100.*

> **Workflow:**
> 1. Attach your dataset (CSV files) as a Kaggle Dataset input.
> 2. Set `DATASET_NAME` below to match the input folder name.
> 3. Run All → download `tanglish-distilgpt2.zip` from the Output panel.

In [ ]:
# ============================================================
#  ✏️  CONFIGURE THIS CELL BEFORE RUNNING
# ============================================================

# Name of your Kaggle Dataset input (the folder under /kaggle/input/).
# e.g. if your dataset is at /kaggle/input/tanglish-data/ → set "tanglish-data"
DATASET_NAME = "tanglish-data"   # <-- change me

# File names inside the dataset folder
TRAIN_FILE = "train.csv"
VAL_FILE   = "val.csv"

# Training hyper-parameters
BLOCK_SIZE      = 128
NUM_EPOCHS      = 4
BATCH_SIZE      = 8
LEARNING_RATE   = 5e-5
WARMUP_RATIO    = 0.06
WEIGHT_DECAY    = 0.01

# Output paths (Kaggle working dir is persisted → downloadable)
OUTPUT_MODEL_DIR = "/kaggle/working/tanglish-distilgpt2"
OUTPUT_ZIP       = "/kaggle/working/tanglish-distilgpt2.zip"

# ============================================================
import os
DATASET_DIR = f"/kaggle/input/{DATASET_NAME}/"
TRAIN_PATH  = os.path.join(DATASET_DIR, TRAIN_FILE)
VAL_PATH    = os.path.join(DATASET_DIR, VAL_FILE)

print(f"Train : {TRAIN_PATH}")
print(f"Val   : {VAL_PATH}")
print(f"Model output: {OUTPUT_MODEL_DIR}")

In [ ]:
# ── Install / upgrade dependencies ──────────────────────────
# Kaggle already has torch; we only upgrade the HF stack.
!pip install -q --upgrade transformers datasets accelerate

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import os, math, zipfile
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Load CSVs ────────────────────────────────────────────────
def load_split(path: str, split_name: str) -> Dataset:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{split_name} file not found: {path}\n"
            "Check DATASET_NAME in the config cell."
        )
    df = pd.read_csv(path)
    if "text" not in df.columns:
        raise ValueError(f"'{split_name}' CSV must have a 'text' column. Found: {list(df.columns)}")
    df = df.dropna(subset=["text"])
    df["text"] = df["text"].astype(str).str.strip()
    print(f"{split_name}: {len(df):,} rows")
    return Dataset.from_pandas(df[["text"]], preserve_index=False)

raw_datasets = DatasetDict({
    "train"     : load_split(TRAIN_PATH, "train"),
    "validation": load_split(VAL_PATH,   "val"),
})
print(raw_datasets)

In [ ]:
# ── Tokenizer ────────────────────────────────────────────────
MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 family has no pad token — reuse eos_token
tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size : {len(tokenizer)}")
print(f"EOS token  : {tokenizer.eos_token!r}  (id {tokenizer.eos_token_id})")
print(f"PAD token  : {tokenizer.pad_token!r}  (id {tokenizer.pad_token_id})")

In [ ]:
# ── Tokenise + group into blocks of BLOCK_SIZE ───────────────
#
# Standard CLM pre-processing:
#   1. Tokenise all texts (no truncation yet)
#   2. Concatenate into one long sequence
#   3. Chunk into non-overlapping blocks of BLOCK_SIZE
# This avoids wasted padding and gives the model full-context windows.

def tokenize_fn(batch):
    return tokenizer(batch["text"])

def group_texts(examples):
    # Concatenate all token lists
    concatenated = {k: sum(examples[k], []) for k in examples}
    total_len = len(concatenated["input_ids"])
    # Drop the remainder so all chunks are exactly BLOCK_SIZE
    total_len = (total_len // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [v[i : i + BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)]
        for k, v in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

tokenized = raw_datasets.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenising",
)
lm_datasets = tokenized.map(
    group_texts,
    batched=True,
    desc="Grouping into blocks",
)

print(lm_datasets)
print(f"Train blocks : {len(lm_datasets['train']):,}")
print(f"Val blocks   : {len(lm_datasets['validation']):,}")

In [ ]:
# ── Load model ───────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in model.parameters())
print(f"DistilGPT2 parameters: {n_params:,}")

In [ ]:
# ── Training ─────────────────────────────────────────────────
use_fp16 = DEVICE == "cuda"

training_args = TrainingArguments(
    output_dir                = OUTPUT_MODEL_DIR,
    overwrite_output_dir      = True,
    num_train_epochs          = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate             = LEARNING_RATE,
    warmup_ratio              = WARMUP_RATIO,
    weight_decay              = WEIGHT_DECAY,
    lr_scheduler_type         = "cosine",
    fp16                      = use_fp16,
    # Eval & checkpointing
    eval_strategy             = "epoch",
    save_strategy             = "epoch",
    save_total_limit          = 1,           # keep only best checkpoint
    load_best_model_at_end    = True,
    metric_for_best_model     = "eval_loss",
    greater_is_better         = False,
    # Logging
    logging_dir               = "/kaggle/working/logs",
    logging_steps             = 50,
    logging_first_step        = True,
    report_to                 = "none",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm       = False,   # Causal LM — no masking
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = lm_datasets["train"],
    eval_dataset  = lm_datasets["validation"],
    tokenizer     = tokenizer,
    data_collator = data_collator,
)

print(f"Training on {DEVICE.upper()}  |  fp16={use_fp16}")
print(f"Steps per epoch : {len(lm_datasets['train']) // BATCH_SIZE}")
train_result = trainer.train()

# Print train / val loss per epoch from log history
print("\n── Loss summary ──────────────────────────")
for log in trainer.state.log_history:
    if "eval_loss" in log:
        epoch = log.get("epoch", "?")
        tr_loss = log.get("loss", "N/A")
        ev_loss = log["eval_loss"]
        ppl = math.exp(ev_loss)
        print(f"  Epoch {epoch:>4}  train_loss={tr_loss}  eval_loss={ev_loss:.4f}  perplexity={ppl:.2f}")

In [ ]:
# ── Save model + tokenizer ───────────────────────────────────
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)
trainer.save_model(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"Model saved to: {OUTPUT_MODEL_DIR}")
print("Files:", os.listdir(OUTPUT_MODEL_DIR))

In [ ]:
# ── Zip for easy download ─────────────────────────────────────
print(f"Zipping {OUTPUT_MODEL_DIR} → {OUTPUT_ZIP} ...")
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_MODEL_DIR):
        # Skip trainer checkpoint subdirs to keep zip small
        dirs[:] = [d for d in dirs if not d.startswith("checkpoint-")]
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, os.path.dirname(OUTPUT_MODEL_DIR))
            zf.write(full_path, arcname)

zip_size_mb = os.path.getsize(OUTPUT_ZIP) / 1e6
print(f"Done. ZIP size: {zip_size_mb:.1f} MB")
print(f"Download it from: Kaggle Output panel → {os.path.basename(OUTPUT_ZIP)}")

In [ ]:
# ── Sanity-check: generate completions for sample prompts ────
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model     = OUTPUT_MODEL_DIR,
    tokenizer = OUTPUT_MODEL_DIR,
    device    = 0 if DEVICE == "cuda" else -1,
)

SAMPLE_PROMPTS = [
    "Naan gym ku pogaren",
    "Avan late ah varuvaan",
    "Ippo enna panre bro",
    "Romba naal aachu da",
]

print("\n── Sample completions ──────────────────────────────────")
for prompt in SAMPLE_PROMPTS:
    out = generator(
        prompt,
        max_new_tokens     = 20,
        do_sample          = True,
        temperature        = 0.8,
        top_p              = 0.9,
        repetition_penalty = 1.2,
        pad_token_id       = tokenizer.eos_token_id,
    )
    generated = out[0]["generated_text"]
    completion = generated[len(prompt):].strip()
    print(f"  Prompt     : {prompt!r}")
    print(f"  Completion : {completion!r}")
    print()